# Session 3 — Real documents, LCEL, and prompt engineering

**Goals**:
1. Ingest a real **PDF** (multi-document setup)
2. Rewrite the chain using **LCEL** (LangChain Expression Language) — the idiomatic `|` pipe syntax
3. Run **prompt engineering** experiments and compare behaviors

**Prerequisites**:
- `uv add pypdf`
- Plus everything from sessions 1–2.
- Ollama running (app open or `ollama serve`)
- **Before you start**: drop any PDF into `data/` — a paper you like, your own resume, anything — and rename it `document.pdf`, or adjust the path below.
  The outputs saved in this notebook were produced with the AWS [*Data Analytics Lens*](https://docs.aws.amazon.com/pdfs/wellarchitected/latest/analytics-lens/analytics-lens.pdf) whitepaper (142 pages); download it yourself if you want to reproduce the same results — it is not included in this repo (Amazon copyright).

## 1. Load and chunk a PDF

New loader, same pattern. `PyPDFLoader` returns one `Document` per page, each with metadata (`source`, `page`).

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "data/document.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages")
print(f"Metadata of page 0: {pages[0].metadata}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # slightly bigger than session 1 — real prose needs more room
    chunk_overlap=80,
)
chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

/var/folders/9g/m2fp6bjs1gs7cx_byglbc3p80000gn/T/ipykernel_38125/3123361699.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 142 pages
Metadata of page 0: {'producer': 'Apache FOP Version 2.6', 'creator': 'ZonBook XSL Stylesheets with Apache FOP', 'creationdate': '2026-07-08T00:32:54+00:00', 'title': 'Data Analytics Lens - AWS Well-Architected Framework', 'author': 'Amazon Web Services', 'keywords': 'data analytics, data analytics lens, analytics, well-architected, whitepaper, analytics lens, workload checklist, data analytics, design principles, well-architected, analytics scenarios, data analytics scenarios', 'source': 'data/document.pdf', 'total_pages': 142, 'page': 0, 'page_label': 'i'}
Created 715 chunks


## 2. Build a fresh vector store

We use a **separate collection** so we don't mix these vectors with session 1's sample.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db",
    collection_name="pdf_docs",   # <- new: named collection
)
print(f"Vectors in 'pdf_docs': {db._collection.count()}")

retriever = db.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors in 'pdf_docs': 715


## 3. The chain, rewritten in LCEL

In session 2 you wrote `rag_chain()` as an explicit function. LCEL expresses the same flow declaratively with the `|` (pipe) operator — each component's output feeds the next one's input.

This is the syntax you'll see in virtually every LangChain codebase, so it's worth internalizing. Compare it mentally with your session 2 function: same three steps, different style.

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model="llama3.2", temperature=0)

prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context. If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# The LCEL chain: retrieve -> format -> prompt -> llm -> plain string
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is this document about?"))

I don't know. The context doesn't provide enough information to determine what the document is about.


### How to read that chain

- `{"context": retriever | format_docs, "question": RunnablePassthrough()}` — runs in parallel: the question goes to the retriever (whose docs get formatted into a string) AND passes through untouched as `question`
- `| prompt` — fills the template with both values
- `| llm` — sends it to llama3.2
- `| StrOutputParser()` — unwraps `response.content` for you

Same Retrieve → Augment → Generate, just declarative.

## 4. Prompt engineering experiments

Now the fun part. We'll keep the chain identical and **only swap the prompt**, observing how behavior changes. This is the core skill of prompt engineering: small text changes, big behavioral differences.

In [9]:
def make_chain(template: str):
    """Helper: build the same chain with a different prompt."""
    p = ChatPromptTemplate.from_template(template)
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | p
        | llm
        | StrOutputParser()
    )

QUESTION = "As a Data Scientist, how can this document help me to use AWS?"  # <- adapt to your PDF

### Experiment A — Baseline vs. persona

In [10]:
baseline = make_chain("""Answer the question based only on the following context.

Context:
{context}

Question: {question}""")

persona = make_chain("""You are a senior technical interviewer. Answer concisely and precisely, based only on the following context. Use bullet points.

Context:
{context}

Question: {question}""")

print("--- BASELINE ---")
print(baseline.invoke(QUESTION))
print("\n--- PERSONA ---")
print(persona.invoke(QUESTION))

--- BASELINE ---
This document can help you as a Data Scientist by providing guidance on designing architectures for analytics applications and environments that follow best practices and strategies recommended by the AWS Well-Architured Framework. Specifically, it may offer insights into:

1. Implementing data platforms that meet governance and compliance requirements.
2. Understanding how to organize your AWS environment using multiple accounts (if relevant to your specific use case).
3. Leveraging the Data Analytics Lens to inform design decisions for analytics applications.

By reading this document, you can gain a better understanding of how to design and implement effective data platforms on AWS that meet the needs of your organization, while also ensuring compliance with governance and regulatory requirements.

--- PERSONA ---
Here are some concise points on how the document can help you as a Data Scientist using AWS:

• Understand the AWS Well-Architected Framework and its prin

### Experiment B — Grounding strictness

Ask something NOT in your PDF and compare a weak prompt vs. a strict one.

In [11]:
OFF_TOPIC = "Who won the World Cup in 2010?"

weak = make_chain("""Use the context to answer.

Context:
{context}

Question: {question}""")

strict = make_chain("""Answer ONLY using the context below. If the answer is not explicitly in the context, reply exactly: "I cannot answer this from the provided documents."

Context:
{context}

Question: {question}""")

print("--- WEAK ---")
print(weak.invoke(OFF_TOPIC))
print("\n--- STRICT ---")
print(strict.invoke(OFF_TOPIC))

--- WEAK ---
There is no information provided in the context about the World Cup or any sports-related topics. The context appears to be related to choosing the best-performing compute, storage, and file solutions for Amazon Web Services (AWS). Therefore, it's not possible to answer the question about who won the World Cup in 2010 based on this context.

--- STRICT ---
I cannot answer this from the provided documents.


### Experiment C — Structured output

Forcing a format — extremely useful when downstream code consumes the answer.

In [12]:
structured = make_chain("""Based only on the context, answer in this exact format:

ANSWER: <one sentence>
CONFIDENCE: <high / medium / low>
SOURCE HINT: <quote max 10 words from the context that supports the answer>

Context:
{context}

Question: {question}""")

print(structured.invoke(QUESTION))

ANSWER: This document will provide guidance on designing architectures for analytics applications using AWS best practices and strategies.
CONFIDENCE: high
SOURCE HINT: "designing architectures" (implies the document provides practical advice)
